In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
import PATH
import my_utils
import re
import numpy as np
import pandas as pd

# metrics
import torch
import torchmetrics
from models.inDecay import Topk_Event_Overlapping

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from qrguide import analysis_fn, transformation

In [4]:
from scipy.stats import pearsonr

pj = os.path.join

all_912_class = my_utils.All_Lindel_class
all_557_class = list(my_utils.rev_index.values())


class_ls = all_557_class
ndel = len(class_ls) - 21
del_frameshift = [int(e.split('+')[1]) for e in class_ls[:ndel]] 
ins_frameshift = [int(e.split('+')[0]) for e in class_ls[ndel:-1]] +[3]
all_frameshift= del_frameshift + ins_frameshift

frameshift_transform = np.array([fs%3!=0 for fs in all_frameshift])

frameshift_transform.shape

# the unified test set
test_oligo_path = pj(PATH.main_dir, "result/test_set_oligo_Feb2.txt")
test_oligos = pd.read_table(test_oligo_path, names=['OligoID'])["OligoID"].values


## read Test set 
experiments =["ST_June_2017_BOB_LV7A_DPI7", "ST_June_2017_BOB_LV7B_DPI7" ,
      "ST_June_2017_CHO_LV7A_DPI7", "ST_June_2017_CHO_LV7B_DPI7",
      "ST_June_2017_E14TG2A_LV7A_DPI7", "ST_June_2017_E14TG2A_LV7B_DPI7",
      "ST_June_2017_HAP1_LV7A_DPI7", "ST_June_2017_HAP1_LV7B_DPI7",
      "ST_June_2017_K562_800x_LV7A_DPI7", "ST_June_2017_K562_800x_LV7B_DPI7",]

celltypes = [exp.split("_")[3] for exp in experiments if 'LV7A' in exp]
celltype_rename = ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562']

In [5]:
rename_dict = {
    'BOB': 'iPSC', 'CHO': 'CHO',
    'E14TG2A': 'mESC', 'HAP1': 'HAP1','K562': 'K562'
}

In [6]:
yulu_test = pd.read_table("/home/wergillius/Project/CRISPR_data/result/oligo.txt", names=['Oligo'])
yulu_test_oligos = yulu_test['Oligo'].values

# ForeCast  Format evaluation fun

## computing  metrics

In [9]:
import pickle as pkl
from scripts.reimplement_forecast import read_data, Forecast_model, ForeCast_dataset
from scripts.reimplement_forecast import find_ckpt

In [10]:
exp = 'ST_June_2017_BOB_LV7A_DPI7'
best_ckpt_path = find_ckpt(pj(PATH.pth_dir, f"ForeCast/{exp}/lightning_logs"))

UnboundLocalError: local variable 'ckpt' referenced before assignment

In [11]:
def find_ckpt(ckpt_version_dir):
    """
    find the latest version, if not finished then return last one
    """
    get_v = lambda s: int(s.replace("version_",""))
    
    versions = [get_v(subdir) for subdir in os.listdir(ckpt_version_dir)]

    for v in versions:
        checkpoint_dir = pj(ckpt_version_dir, 'version_%d'%v, 'checkpoints')
        if not os.path.exists(checkpoint_dir):
            try:
                shutil.rmtree(pj(ckpt_version_dir, 'version_%d'%v)) 
            except:
                continue


    versions = [get_v(subdir) for subdir in os.listdir(ckpt_version_dir)]
    maxv  = np.max(versions)

    version_dir = pj(ckpt_version_dir, 'version_%d'%maxv, 'checkpoints')

    try:
        ckpts = os.listdir()
    except FileNotFoundError:
        maxv -= 1
    ckpts = list(filter(lambda x : x.endswith('.ckpt'), 
                            os.listdir(version_dir))
                )
    
    while len(ckpts) == 0:
        maxv -= 1
        ckpts = list(filter(lambda x : x.endswith('.ckpt'), 
                            os.listdir(version_dir))
                            )
    
    if len(ckpts) >1:

        for ct in ckpts:
            if ct.replace(".ckpt","TestPred.pkl") in os.listdir(version_dir):
                ckpt = ct

    else:
        ckpt = ckpts[0]
        
    
    return pj(ckpt_version_dir, 'version_%d'%maxv, 'checkpoints', ckpt)

# **ST DeepDecay**

In [12]:
import pickle as pkl

In [13]:
from importlib import reload

In [14]:
from scripts import reimplement_forecast

In [15]:
exp = 'ST_June_2017_CHO_LV7A_DPI7'
find_ckpt(pj(PATH.pth_dir, f"ST_featv2_ST_DeepDecay_interaction/{exp}/lightning_logs")).replace(".ckpt","TestPred.pkl")

'/home/wergillius/data/CRISPR_data/pl_trainer_log/ST_featv2_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=39-val_loss=0.00000000TestPred.pkl'

In [28]:
methods = ['ForeCast', 'ST_featv2_ST_Decay_Scaler_interaction', 'ST_featv2_ST_DeepDecay_interaction']
model_name = ['ForeCast', 'inDecay', 'DeepDecay']

performance_df = pd.DataFrame([])

for i,  method in enumerate(methods):

    performance_dict = {}

    for exp in experiments:
        if "LV7B" in exp:
            continue
            
        cell = exp.split("_")[3]
        cell = rename_dict[cell]
            
        Y1_pkl_path = f'{PATH.pth_dir}/{method}/{exp}/ForeCast_TestY.pkl'
        f = open(Y1_pkl_path, 'rb')
        Y1_lookup = pkl.load(f)  # forecast : ST
        f.close()
        
        
        # read the prediction
        pred_Y_path = find_ckpt(f'{PATH.pth_dir}/{method}/{exp}/lightning_logs').replace(".ckpt","TestPred.pkl")
        assert os.path.exists(pred_Y_path)
        g = open(pred_Y_path, 'rb')
        pred_lookup = pkl.load(g)
        g.close()

        
        performance_json = analysis_fn.assessment_recipe_forecast(Y1_lookup, pred_lookup)
        IDL_performance = analysis_fn.assessment_recipe_IDL_forecast(Y1_lookup, pred_lookup)

        performance_json.update(IDL_performance)

        performance_dict[cell] = performance_json

    Dpdcy_perform_df = pd.json_normalize(list(performance_dict.values()))
    Dpdcy_perform_df['celltype'] = celltype_rename
    Dpdcy_perform_df['model'] = model_name[i]

    performance_df = pd.concat([performance_df, Dpdcy_perform_df], axis=0)



In [26]:
performance_df

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,celltype,model
0,1.096283,3.045896,5.641659,0.785945,3.140335,5.778464,iPSC,ForeCast
1,0.709189,3.313327,6.711386,0.709750,3.448367,6.932921,CHO,ForeCast
2,0.836088,3.203883,6.386584,0.804116,3.344219,6.563107,mESC,ForeCast
3,0.761291,3.253310,6.231244,0.795160,3.358341,6.404237,HAP1,ForeCast
4,0.757863,3.286849,6.426302,0.773155,3.337158,6.569285,K562,ForeCast
0,1.261741,2.834951,5.357458,0.738536,3.060900,5.714034,iPSC,inDecay
1,0.847064,2.915269,6.225066,0.660368,3.151809,6.617829,CHO,inDecay
2,0.972187,3.015887,6.014122,0.755460,3.215357,6.341571,mESC,inDecay
3,0.882602,2.820830,5.610768,0.710619,3.022948,6.015004,HAP1,inDecay
4,0.859111,2.886143,5.911739,0.727728,2.997352,6.218005,K562,inDecay


In [27]:
performance_df.to_csv("../result/benchmarking/CollapseI_performance_Sep17.csv", index=False)

In [50]:
Dpdcy_perform_df.to_csv("../result/benchmarking/Dpdcy_performance_Jun17.csv", index=False)

## plotting

In [4]:
performance_df = pd.read_csv("../result/benchmarking/Benchmarking_result_Jun17.csv")

In [17]:
def rgb_to_hex(rgb):
    # Ensure that each color value is within the valid range of 0-255
    r, g, b = [min(255, max(0, c)) for c in rgb]
    # Convert the RGB values to a hex string and return it
    return f"#{r:02x}{g:02x}{b:02x}"
model_palette  = {
    'Lindel': rgb_to_hex([103, 126, 152]), 
    'inDecay': rgb_to_hex([166, 152, 161]), 
    'DeepDecay': rgb_to_hex([206, 172, 45]), 
    'FORECasT': rgb_to_hex([194, 124, 68]), 
    'replication': rgb_to_hex([156, 147, 118]), 
}
model_palette = {'Lindel': '#677e98',
 'inDecay': '#a698a1',
 'DeepDecay': '#ceac2d',
 'FORECasT': '#c27c44',
 'replication': '#9c9376'}

In [19]:
metrics_to_show = ['Top5 events recall', 'Top10 events recall',
                'Top5_IDL','Top10_IDL',
                'KLD_IDL', 'Kendall_tau_IDL',
                'R2 of Frameshift ratio',  'delratio_r2'
                ]

# Test

In [30]:
BOB_50 = np.load('/home/wergillius/Project/CRISPR_data/result/yulu_random_oligo_list/BOB_50_finetune_list.npy')

In [31]:
BOB_50

array(['Oligo13187', 'Oligo1448', 'Oligo1466', 'Oligo35360', 'Oligo51006',
       'Oligo51285', 'Oligo48595', 'Oligo47287', 'Oligo8754',
       'Oligo22335', 'Oligo12641', 'Oligo20779', 'Oligo34034',
       'Oligo18471', 'Oligo17283', 'Oligo4943', 'Oligo45127',
       'Oligo36640', 'Oligo20310', 'Oligo32216', 'Oligo39237',
       'Oligo9428', 'Oligo8908', 'Oligo41428', 'Oligo42047', 'Oligo18199',
       'Oligo19943', 'Oligo7717', 'Oligo22712', 'Oligo47057', 'Oligo5362',
       'Oligo29927', 'Oligo32410', 'Oligo10605', 'Oligo26417',
       'Oligo6257', 'Oligo29743', 'Oligo34767', 'Oligo30337', 'Oligo5464',
       'Oligo1439', 'Oligo6728', 'Oligo29330', 'Oligo34377', 'Oligo41784',
       'Oligo3980', 'Oligo5859', 'Oligo7462', 'Oligo44657', 'Oligo39399'],
      dtype='<U10')